# Which genetic perturbations show similar effects?

This notebook answers the seminar's third task (numbered `03_` in this folder, following on from
`02_condition_classification.ipynb`): **using only the `Control` condition cells**, do different
CRISPR knockouts of the 248 target genes in this Perturb-seq screen produce similar transcriptional
effects, and which unsupervised clustering method best recovers real structure between them?

As in Task 2, all the computation lives in standalone scripts under `perturbation_clustering/`
(`build_signatures.py`, `cluster_and_visualize.py`) so this notebook just reads back
`perturbation_clustering/results/` and discusses it.

**Why Control only, as instructed:** restricting to Control condition cells means every knockout's
effect is measured against the *same* non-targeting baseline, in the *same* experiment/batch — there is
no IFNγ stimulus or co-culture confound layered on top. Any similarity we find between two
perturbations' Control-condition signatures reflects their baseline/tonic transcriptional effect on
unstimulated melanocytes.

In [ ]:
import json

import pandas as pd
from IPython.display import Image, display

RESULTS_DIR = "perturbation_clustering/results"
FIG_DIR = f"{RESULTS_DIR}/figures"
pd.set_option("display.max_colwidth", 200)

## 1. Building per-perturbation signatures (`build_signatures.py`)

For each of the 249 perturbation targets (248 genes + non-targeting `control`), restricted to the
57,627 Control-condition cells, we compute a **pseudobulk signature**:

$$\text{signature}[p, g] = \text{mean}_{\text{cells in }p}(\text{log1p-norm expr of }g) -
\text{mean}_{\text{non-targeting control cells}}(\text{log1p-norm expr of }g)$$

on the same 2000 HVGs used throughout this project. Subtracting the non-targeting control's mean turns
each knockout's profile into a background-corrected *effect* vector, comparable across perturbations,
rather than raw expression levels (which mostly reflect shared cell state, not the knockout).

**Filtering**: perturbations profiled in fewer than 15 Control cells are dropped — their pseudobulk
mean would be dominated by cell-to-cell sampling noise, not a real average effect. This drops 9 of the
249 targets: `TUBB` (2 cells), `UBL5` (3), `PSMA7` (4), `RACK1` (7), `ATP1A1`/`UBA52` (10 each),
`SNRPF`/`SNRPE` (11 each), `CCT6A` (14) — every one of them a core translation/spliceosome/proteasome/
chaperone gene, consistent with strong essential-gene fitness effects depleting those cells from the
pool before sequencing. **239 perturbations** remain for clustering.

In [ ]:
counts = pd.read_csv(f"{RESULTS_DIR}/perturbation_cell_counts.csv", index_col=0)
sig = pd.read_csv(f"{RESULTS_DIR}/signatures.csv", index_col=0)
print(f"{counts.shape[0]} perturbations profiled in Control condition, {counts['n_cells'].sum()} cells total")
print(f"{sig.shape[0]} perturbations x {sig.shape[1]} HVGs kept for clustering (>=15 cells)")
counts.sort_values("n_cells").head(10)

## 2. An initial look: the signature correlation map

Before doing any clustering, the simplest possible question is: how correlated are two perturbations'
signatures with each other, across the 2000 HVGs? This is just Pearson correlation, computed directly
on the raw signatures — no distance metric or standardization choice enters yet.

In [ ]:
display(Image(filename=f"{FIG_DIR}/signature_correlation_map.png"))

Two things are visible immediately: **most perturbations are mildly-to-strongly positively
correlated with each other** (the broad red block covering most of the matrix), and there is a **small
group of perturbations in the bottom-right that are clearly *anti*-correlated** with the rest (the blue
band). We'll come back to exactly which genes make up that anti-correlated group in Section 5 — it turns
out to be the same essential-gene group flagged by the low-cell-count filter above.

## 3. Raw magnitude vs. row-normalized signatures — and the math

`sig` already has a well-defined pairwise correlation structure (Section 2). But **correlation is not
what ordinary clustering algorithms operate on** — hierarchical (Ward) and k-means both minimize
**Euclidean distance**, and Euclidean distance is sensitive to a vector's magnitude, not just its
direction. Some perturbations have a *much* larger raw signature than others:

In [ ]:
display(Image(filename=f"{FIG_DIR}/magnitude_vs_normalized_clustering.png"))

**Left panel**: the top 20 perturbations by raw signature L2 norm are all core-essential genes —
`EIF2S3`, `SAE1`, `NACA`, `AHCY`, `DNAJC9`, `RUVBL2`, ... (translation, proteasome, spliceosome,
chaperone machinery). Knocking out something essential to survive without causes a much *bigger*
transcriptional shock than knocking out, say, a surface receptor — that's a difference in **magnitude**,
not necessarily in **kind**.

**Middle panel**: clustering directly on signatures that have only been column- (per-gene-) standardized
— i.e. magnitude differences between perturbations are left intact — at a small, illustrative k=4 gives
cluster sizes **223 / 14 / 1 / 1**. Almost everything lands in one undifferentiated cluster, while the
essential-gene outliers peel off one or two at a time. This is Euclidean distance doing exactly what
it's supposed to do — but it's answering "which perturbations have an unusually large effect," not
"which perturbations have a *similar* effect."

**Right panel**: the fix is to **row- (per-perturbation-) standardize** each signature — subtract its
own mean and divide by its own standard deviation across the 2000 genes — before computing distance.
The same Ward clustering at k=4 now gives a much more balanced **105 / 67 / 46 / 21** split.

**Why this works, mathematically**: let $a$ and $b$ be two perturbations' *row-standardized* signatures
(each has mean 0 and variance 1 across the $n{=}2000$ genes, so $\|a\|^2 \approx n$ and $\|b\|^2
\approx n$). Then

$$\|a - b\|^2 = \|a\|^2 + \|b\|^2 - 2\,a\!\cdot\!b = 2n - 2n\cdot\text{corr}(a,b) = 2n\,(1 - \text{corr}(a,b))$$

because $a \cdot b = n \cdot \text{corr}(a, b)$ for mean-0/variance-1 vectors. Squared Euclidean distance
after row-standardizing is therefore an **exact, monotonically decreasing function of Pearson
correlation** between the *original* raw signatures. In other words: row-standardizing before clustering
makes Euclidean-distance-based methods (Ward, k-means) cluster on exactly the correlation structure we
already looked at in Section 2 — by construction, blind to each perturbation's overall effect size. This
is what every clustering method from here on is run on.

In [ ]:
import numpy as np

sig_vals = sig.values
row_std = (sig_vals - sig_vals.mean(1, keepdims=True)) / (sig_vals.std(1, keepdims=True) + 1e-8)

# spot-check the derivation above on two arbitrary perturbations
i, j = 0, 1
a, b = row_std[i], row_std[j]
lhs = np.sum((a - b) ** 2)
corr_ab = np.corrcoef(sig_vals[i], sig_vals[j])[0, 1]  # correlation of the RAW signatures
rhs = 2 * len(a) * (1 - corr_ab)
print(f"{sig.index[i]} vs {sig.index[j]}:  ||a-b||^2 = {lhs:.1f}   2n(1-corr) = {rhs:.1f}")

## 4. Three clustering methods, and how each one's cluster count was chosen

All three methods run on the same row-standardized 239 × 2000 signature matrix (reduced to 30 PCs):

- **Hierarchical (Ward)** and **k-means** — both scanned over k = 4–25, cluster count chosen by
  whichever k maximizes the silhouette score.
- **Leiden** — a k-NN graph is built over the perturbations (same machinery used for cell clustering
  in Task 1, applied here to "cells" = perturbations) and resolution is scanned over {0.3, ..., 2.0};
  silhouette again picks the resolution.

In [ ]:
with open(f"{RESULTS_DIR}/k_selection_scores.json") as f:
    sel = json.load(f)
print("Chosen k / resolution:", sel["chosen_k"], "| leiden resolution:", sel["chosen_leiden_resolution"])
display(Image(filename=f"{FIG_DIR}/silhouette_selection.png"))

Hierarchical and k-means both peak sharply at **k=4** and then wander in a low, noisy band as k
increases — a real, if modest, interior optimum. **Leiden's silhouette instead decreases
monotonically** across the whole scanned resolution range, so silhouette pushes it to the coarsest
partition tested, **resolution=0.3 → k=2**. This isn't a bug: it's a known property of internal metrics
like silhouette — they tend to reward fewer, larger, well-separated clusters, and a 2-way split is the
easiest one to make well-separated. Read Leiden's k=2 as "the most confident single split the graph
supports," not as "the graph naturally has exactly 2 communities."

As it turns out, that 2-way split is informative rather than arbitrary — see Section 5.

## 5. Visualization: UMAP of the three partitions

One shared UMAP embedding (built on the same row-standardized/PCA signatures), colored by each method's
own cluster labels — so differences between panels are purely about how each method draws its
boundaries, not about the underlying geometry.

In [ ]:
display(Image(filename=f"{FIG_DIR}/umap_clustering_comparison.png"))

The small cluster in the bottom-left (teal in all three panels) is picked out consistently by
every method, including as Leiden's entire k=2 split — this is the essential-gene/growth-arrest group
from Sections 2–3. Hierarchical and k-means agree closely on how they subdivide the remaining, much
larger group into three further parts (matching regions in both panels almost exactly); Leiden's
silhouette-optimal resolution doesn't subdivide it further.

## 6. Biological interpretation — what's actually in each cluster?

Below is the full gene membership of every cluster, for all three methods, read directly from
`cluster_labels.csv` — no external pathway database was used; the interpretation below is reasoned
directly from each gene's known function.

In [ ]:
labels = pd.read_csv(f"{RESULTS_DIR}/cluster_labels.csv", index_col=0)

for method in ["hierarchical", "kmeans", "leiden"]:
    print(f"===== {method} =====")
    sizes = labels[method].value_counts().sort_values(ascending=False)
    for c in sizes.index:
        genes = sorted(labels.index[labels[method] == c])
        print(f"--- cluster {c} (n={len(genes)}) ---")
        print(", ".join(genes))
        print()

**The essential-gene / growth-arrest cluster (hierarchical cluster 3, n=21; k-means cluster 3,
n=22; leiden cluster 1, n=17) — the clearest, most consistent finding in this analysis.** All three
methods agree almost gene-for-gene on this group: `EEF1G`, `EIF4A1`, `FARSA`, `PABPC1` (translation),
`NCL`, `PUF60`, `SNRPG` (spliceosome/nucleolar), `CTPS1`, `PAICS`, `PPA1`, `GPI` (nucleotide/energy
metabolism), `MYC`, `CCND1`, `CDK6` (cell-cycle drivers), `UQCRFS1` (OXPHOS), `CCT3`, `DNAJC9`
(chaperones). **This makes strong biological sense**: knocking out core machinery that a cell needs to
divide and translate protein triggers a shared, generic growth-arrest/integrated-stress response — a
real, coherent shared "effect," just not a specific pathway in the way MHC-I or JAK-STAT would be. It is
also exactly the group that dominates raw-magnitude clustering (Section 3) and forms the anti-correlated
block in the correlation map (Section 2): three completely different analyses converge on the same 17–22
genes.

**The larger, further-subdivided group** (everything else — Leiden stops here at k=2, but hierarchical/
k-means split it three ways) is a mixed bag with **partial, not perfect, biological coherence**:

- Hierarchical cluster 1 / k-means cluster 0 contains `HLA-A`, `HLA-B`, `HLA-E`, `HLA-H`, `HLA-DRB5`
  (MHC class I/II), `IFNGR2`, `JAK1`, `JAK2`, `STAT1`, `STAT3`, `TRIM22` (JAK-STAT signal transducers),
  and separately `MC1R`, `MIA`, `KLF4`, `SOX4` (melanocyte identity) and `CDKN1A`, `CDKN2A`, `CDKN2B`,
  `CCND2` (cell-cycle regulators). Grouping MHC-I structural genes with the JAK-STAT genes that induce
  them is sensible; melanocyte-identity and cell-cycle genes landing in the same bucket is more likely
  this coarse a partition lumping together several *different*, weaker signals rather than one shared
  mechanism.
- Hierarchical cluster 2 / k-means cluster 1 contains a *different* subset of innate-immune genes —
  `CGAS`, `IRF3`, `TMEM173` (STING), `RTP4`, `SP100`, `RNF213`, `IFNGR1` — plus `TAPBP`/`TAPBPL`
  (MHC-I peptide loading). **Notably, this splits the IFN/JAK-STAT axis across two different clusters**
  (`JAK1`/`JAK2`/`STAT1`/`STAT3`/`IFNGR2` in cluster 1 vs. `CGAS`/`TMEM173`/`IFNGR1` here) — a
  plausible finer distinction (receptor/transcription-factor genes vs. cytosolic-sensor/effector genes),
  but just as plausibly this coarse a partition arbitrarily dividing one broader innate-immune signal in
  half. We can't fully tell the two apart from a k=4 partition alone.
- Hierarchical cluster 4 / k-means cluster 2 contains `E2F1`, `FOXM1`, `CDK4`, `DNMT1`, `POLD2`
  (DNA-replication/cell-cycle-progression genes), plus `HLA-C`, `HLA-F` — **the two remaining MHC-I
  alleles, split off from `HLA-A`/`HLA-B`/`HLA-E`/`HLA-H` in cluster 1.** Splitting one gene family
  (MHC-I) across two different clusters is a clear example of where this clustering does *not* fully
  recover known biology, and is worth stating plainly rather than glossing over.

**Overall read**: the one really clean, method-independent result is the essential-gene/growth-arrest
group. Beyond that, hierarchical and k-means agree closely with *each other* but only partially recover
known pathway relationships (e.g. MHC-I alleles split across clusters; the IFN/JAK-STAT axis divided in
a plausible-but-unconfirmed way), and Leiden's silhouette-optimal resolution is too coarse to say
anything beyond "essential genes vs. everything else." None of the three methods cleanly separates a
richer set of specific pathway modules at the cluster counts silhouette selects — which is a reasonable,
honest result at this sample size (239 pseudobulk profiles, many built from only a few dozen cells) and
signature dimensionality, not a failure of the pipeline.

## 7. Summary

1. **The signature correlation map** (Section 2) shows most Control-condition knockouts are mildly
   positively correlated with each other, with one clear anti-correlated group standing out.
2. **That anti-correlated group is a set of core-essential genes** (translation, spliceosome,
   chaperone, OXPHOS, cell-cycle-driver knockouts — `EEF1G`, `EIF4A1`, `FARSA`, `MYC`, `CCND1`, `CDK6`,
   `CCT3`, `UQCRFS1`, `SNRPG`, and others), and it is the single most robust finding in this analysis:
   it dominates raw-magnitude clustering, forms the anti-correlated block in the correlation map, is
   recovered nearly gene-for-gene by all three clustering methods, and is exactly the low-cell-count
   group flagged during signature building. Multiple independent angles on the data agree.
3. **Row-normalizing (per-perturbation z-scoring) before clustering is what makes the rest of the
   analysis meaningful.** Raw-magnitude clustering just re-discovers "which knockouts have a big
   effect" (223/14/1/1 split, Section 3); row-standardizing makes Euclidean distance an exact function
   of Pearson correlation (derived and numerically verified in Section 3) and produces a much more
   balanced, interpretable partition (105/67/46/21).
4. **Hierarchical (Ward) and k-means are the more useful methods here** — both find the same clean
   interior silhouette peak at k=4 and agree closely with each other on cluster membership. **Leiden's
   silhouette has no interior peak** and pushes it to the coarsest possible split (k=2), which still
   correctly isolates the essential-gene group but says nothing more specific.
5. **Biological coherence beyond the essential-gene group is partial**: some real pathway logic is
   visible (MHC-I genes grouping with JAK-STAT signal transducers; a plausible split between
   receptor/transcription-factor and sensor/effector innate-immune genes), but it is not clean — the
   MHC-I gene family itself ends up split across two different clusters. This should be read as an
   honest, modest result given the sample size (239 profiles, several with fewer than 30 cells) rather
   than either "the methods found nothing" or "the methods fully recovered pathway structure." 